In [1]:
import ast
import os

In [2]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [3]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [4]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [5]:
# tree = ast.parse(files["src\\services\\tutor_service.py"])
# print(ast.dump(tree, indent=2))

def parse_file(source_code):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.name for alias in node.names])

        if isinstance(node, ast.ClassDef):
            classes.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
            
        if isinstance(node, ast.FunctionDef):
            functions.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
                
    return {"import_modules": import_modules, "import_names": import_names, "classes": classes, "functions": functions}

In [6]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")


def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()

                    parsed_structure = {"import_modules": [], "import_names": [], "classes": [], "functions": []}

                    if name.endswith(".py"):
                        parsed_structure = parse_file(content)

                    parsed_structure['content'] = content

                    inventory[os.path.join(rel_dir, name)] = parsed_structure
                        

    return inventory

In [7]:
# get_structure(dir)
files = get_files(directory=dir)

In [8]:
for imports in files['src\services\diagnoser_service.py']:
    print(imports)

import_modules
import_names
classes
functions
content


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_4516\1899480365.py:1: SyntaxWarning: invalid escape sequence '\s'
  for imports in files['src\services\diagnoser_service.py']:


In [9]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = {filepath.replace("\\", "/"): data for filepath, data in files.items()}
        self.name_index = self.build_name_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp or not imp.startswith("src."): ## harcoded!
                    continue

                target = imp.replace(".", "/") + ".py" ## hardcoded!
                if target in self.files:
                    imports.append(target)
                else:
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = imports
        return cleared_files

    def get_node_style(self, filepath):
        """Returns a more premium color palette and node properties."""
        if "services" in filepath:
            color = {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"}
        elif "models" in filepath:
            color = {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"}
        elif "database" in filepath:
            color = {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"}
        else:
            color = {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"}
        
        return color

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(
            uri, auth=(user, password)
        )
        
        repo_id = filename

        with driver.session() as session:
            for node in self.G.nodes():
                imports = [
                    name for names in self.files[node]['import_names'] 
                    for name in names if name in self.name_index
                ]

                query = '''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                '''
                session.run(
                    query,
                    name=node.split('/')[-1],
                    path=node,
                    classes=[c['name'] for c in self.files.get(node, {}).get("classes", [])],
                    imports=imports,
                    functions=[f['name'] for f in self.files.get(node, {}).get("functions", [])],
                    repo_id=repo_id
                )
                
            for source, target in self.G.edges():
                query = '''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                '''
                session.run(query, source=source, target=target, repo_id=repo_id)
                

    def show(self, filename="graph.html"):
        net = Network(
            directed=True, 
            notebook=True, 
            cdn_resources='remote', 
            height="750px", 
            width="100%", 
            bgcolor="#111827",
            font_color="white"
        )
        
        net.from_nx(self.G)

        options = {
            "edges": {
                "color": {"inherit": "from", "opacity": 0.5},
                "smooth": {"type": "continuous", "roundness": 0.5},
                "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}},
                "width": 1.5
            },
            "physics": {
                "forceAtlas2Based": {
                    "gravitationalConstant": -60,
                    "centralGravity": 0.005,
                    "springLength": 150,
                    "springStrength": 0.08,
                    "damping": 0.4,
                    "avoidOverlap": 0.5
                },
                "maxVelocity": 50,
                "minVelocity": 0.1,
                "solver": "forceAtlas2Based",
                "stabilization": {"enabled": True, "iterations": 1000}
            },
            "interaction": {
                "hover": True,
                "navigationButtons": True,
                "multiselect": True,
                "tooltipDelay": 200
            }
        }
        
        net.set_options(json.dumps(options))
        
        return net.write_html(filename)


In [10]:
builder = BuildGraph(files)
builder.build()
builder.show()

In [34]:
builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

In [11]:
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')

class VectorStore:
    def __init__(self, files:dict, path='../chromadb', collection_name:str='database'):
        self.files = files
        self.client = chromadb.PersistentClient(path=path)
        ef = OpenAIEmbeddingFunction(api_key=openai_key, model_name="text-embedding-3-small")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=ef)

    def build(self, max_module_lines=100, overlap=10):
        self.chunks = []
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']

            for i in range(0, len(lines), max_module_lines):
                start = max(0, i - overlap)
                end = min(len(lines), i + overlap + max_module_lines)
                chunk_lines = lines[start : end]
                self.chunks.append([
                    "\n".join(chunk_lines),
                    {
                        "path": file,
                        "filename": file.split("/")[-1],
                        "function_name": "module_level_segment",
                        "class_name": "module_level",
                        "line_start": start + 1,
                        "line_end": end
                    }
                ])

            for func in functions:
                parent = 'module_level'
                for c in classes:
                    if func['line_start'] >= c['line_start'] and func['line_start'] <= c['line_end']:
                        parent = c['name']
                        break
                
                start = max(0, func['line_start'] - overlap)
                end = min(len(lines), func['line_end'] + overlap)
                chunk = "\n".join(lines[start:end])
                metadata = {
                    "path": file.replace(),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": parent,
                    "line_start": start + 1,
                    "line_end": end
                }
                self.chunks.append([chunk, metadata])
                self.push()

    def push(self):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]

        ids = [f"{m['path']}:{m['function_name']}:{i}" for i, m in enumerate(metadatas)]

        self.collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )
        print(f"Successfully pushed {len(documents)} chunks to collection '{self.collection.name}'.")


    def search(self, query: str, top_k: int = 5):
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results

In [12]:
client = VectorStore(files, collection_name='prototype')
# client.build()

In [32]:
# client.push()

In [13]:
results = client.search(
    query="what is the function tutor is doing and how many files it is importing from"
)
# (results['ids'])


In [14]:
results

{'ids': [['src\\services\\tutor_service.py:__init__:76',
   'app.py:module_level_segment:1',
   'app.py:module_level_segment:0',
   'src\\services\\tutor_service.py:module_level_segment:71',
   'README.md:module_level_segment:3']],
 'embeddings': None,
 'documents': [['from src.services.tutor_lesson_utils import (\n    PROCEED_PROMPT,\n    format_equations_for_prompt,\n    is_new_topic_intent,\n    is_proceed_to_quiz,\n)\n\n\nclass TutorWorkflow:\n    def __init__(self, neo4j_conn: Neo4jConn):\n        self.neo4j_conn = neo4j_conn\n        self.workflow = self._build_workflow()\n        self.app = self.workflow.compile(checkpointer=MemorySaver())\n        self.llm = ChatGroq(\n            model="llama-3.3-70b-versatile", \n            temperature=0.3, \n            max_tokens=250\n        )\n        # Longer explanations for the teach-first phase\n        self.teach_llm = ChatGroq(\n            model="llama-3.3-70b-versatile",\n            temperature=0.35,\n            max_tokens=1000

In [54]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

groq_api_key = os.getenv('GROQ_API_KEY')

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph'] = Field(..., description="Graph for only relationships between nodes/files and hybrid for both graph and vector similarity search")
    reasoning: str = Field(..., description="Brief reasoning behind the routing decision")
    confidence_score: float = Field(..., description="Confidence score of the decision")

class QueryEngine:
    def __init__(self, repo_id: str, db_client: VectorStore, uri:str, user:str, password:str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))

    def vector_search(self, query: str, filenames: list[str], top_k: int = 5):
        collection = self.db_client.collection
        
        return collection.query(
            query_texts=[query],
            n_results=top_k,
            where={"path": {"$in": filenames}}
        )

    def graph_search(self, query):
        structured_llm = self.llm.with_structured_output(CypherQuery)

        prompt = ChatPromptTemplate.from_messages([
            ("system", """
            You are a cypher query master. Generate a cypher query on the basis of given prompt
            strictly following the structured output schema.
             
            SCHEMA:
            - Central Node: Repo (property: repo_id)
            - Relationship: (:Repo)-[:CONTAINS]->(:File)
            - Relationship: (:File)-[:IMPORTS]->(:File)
            
            CRITICAL RULES:
            1. Your root node is 'Repo'.
            2. The property name is 'repo_id'.
            3. Start every query with: MATCH (r:Repo {{repo_id: '{repo_id}'}})
            
            Example:
            MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File {{name: 'tutor_service.py'}})
            MATCH (f)-[:IMPORTS]->(imported)
            RETURN f.path, imported.path
            """),
            ("user", "Question: {query} \nRepo ID: {repo_id}")
        ])

        chain = prompt | structured_llm

        response = chain.invoke({
            "query": query,
            "repo_id": self.repo_id
        })

        with self.graph_driver.session() as session:
            result = session.run(response.cypher)
            output = [record.data() for record in result]

        return output
    
    def router(self, query: str) -> RouterDecision:
        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """
                You are a query router for a code repository assistant.
                Classify the query into one of three types:

                - graph: query asks about file relationships, imports, dependencies, structure
                - hybrid: query asks about all, structure, logic, function behavior, implementation detailsand implementation

                Return your decision as structured output.
            """),
            ('user', "query: {query}")
        ])

        router_chain = router_prompt | router_llm

        response = router_chain.invoke({"query": query})

        return response
    
    def get_result(self, query: str, top_k: int):
        decision = self.router(query)

        route = decision.decision
        reasoning = decision.reasoning
        score = decision.confidence_score

        if route == "graph":
            return {
                "type": "graph",
                "graph": self.graph_search(query),
                "reasone": reasoning,
                "confidence": score
            }

        else:
            graph_data = self.graph_search(query)
            filenames = list(set([
                v
                for r in graph_data
                for v in r.values()
                if v is not None
            ]))

            if not filenames:
                vector_data = self.db_client.collection.query(
                    query_texts=[query],
                    n_results=top_k
                )
            else:
                vector_data = self.vector_search(query, filenames=filenames, top_k=top_k)

            return {
                "type": "hybrid",
                "graph": graph_data,
                "vector": vector_data,
                "confidence": score
            }
            
    def close(self):
        self.graph_driver.close()
    

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    max_tokens=200,
    api_key=groq_api_key
)
openai_llm = ChatOpenAI(model="gpt-4o", temperature=0)

engine = QueryEngine(repo_id=filename, db_client=client, uri="bolt://localhost:7687", user="neo4j", password="pk142145", llm=llm)
query = "How is app.py working?"
result = engine.get_result(query, 6)

In [56]:
result

{'type': 'hybrid',
 'graph': [{'f.path': 'app.py', 'imported.path': 'src/models/concept.py'},
  {'f.path': 'app.py', 'imported.path': 'src/models/state.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/pdf_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/llm_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/graph_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/intent_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/planner_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/diagnoser_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/tutor_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/database/student_db.py'},
  {'f.path': 'app.py', 'imported.path': 'src/database/neo4j_conn.py'}],
 'vector': {'ids': [['app.py:module_level_segment:1',
    'app.py:module_level_segment:0',
    'app.py:module_level_segment:2']],
  'embeddings': None,
  'documents': [['\n      

In [57]:
def formate_documents(data: dict) -> str:
    context = ""

    if data['type'] in ('graph', 'hybrid'):
        graph_data = data['graph']
        context += f"Relationships:\n"
        for r in graph_data:
            context += f" {r.get('f.path', '')} -> {r.get('imported.path', '')} \n"
        
    if data['type'] == 'hybrid':
        docs = data['vector']['documents'][0]
        metas = data['vector']['metadatas'][0]
        context += "CHUNK DATA:\n"
        for i, docs in enumerate(docs):
            context += f"\n[{metas[i]['path']} | lines {metas[i]['line_start']}-{metas[i]['line_end']}]\n{docs}\n"

    return context
        

In [58]:
formated_result = formate_documents(result)

In [ ]:
from pydantic import BaseModel

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")

class AnswerEngine:
    def __init__(self, llm):
        self.llm = llm.with_structured_output(ResponseModel)

    def generate_response(self, query:str, context: str):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
                You are a code repository assistant.
                Answer only from the provided context.
                Sources should be file names used to answer.
                Confidence should be between 0 and 1.
            """), ("user", "Context:\n{context}\n\nQuestion: {query}")
        ])

        chain = prompt | self.llm

        response = chain.invoke({"context": context, "query": query})

        return response


In [72]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4o", temperature=0)

query = "what if i delete concept.py"
chat_engine = AnswerEngine(openai_llm)
result = chat_engine.generate_response(query, formated_result)

In [73]:
result

ResponseModel(answer='If you delete `concept.py`, the application will likely encounter errors related to missing imports or undefined classes. Specifically, the `Concept` class is imported from `src.models.concept` in `app.py`, and it seems to be used for handling concepts extracted from the learning material. Without this file, any functionality relying on the `Concept` class will fail, potentially causing the application to crash or behave unexpectedly when processing learning content or managing concepts.\n\nTo resolve this, you would need to restore `concept.py` or ensure that any dependencies on it are removed or replaced with alternative implementations.', score=0.9, sources='app.py')

## RAG Evaluation metrices

### Reranking

In [74]:
# questions
eval_questions = [

    # --- GRAPH / STRUCTURAL ---
    {
        "question": "What files does app.py import?",
        "ground_truth": "app.py imports Concept, GraphState from src/models, PDFService, LLMService, GraphService, IntentService, PlannerService, DiagnoserService, TutorWorkflow from src/services, and StudentDB, Neo4jConn from src/database"
    },
    {
        "question": "Which files import Neo4jConn?",
        "ground_truth": "app.py and src/services/tutor_service.py import Neo4jConn"
    },
    {
        "question": "What does tutor_service.py import?",
        "ground_truth": "tutor_service.py imports Neo4jConn, StudentDB, Question, GraphState, DiagnoserService, IntentService, PlannerService, and tutor_lesson_utils"
    },
    {
        "question": "Which files are in the src/database directory?",
        "ground_truth": "src/database contains student_db.py and neo4j_conn.py"
    },
    {
        "question": "Which files are in the src/services directory?",
        "ground_truth": "src/services contains pdf_service.py, llm_service.py, graph_service.py, intent_service.py, planner_service.py, diagnoser_service.py, tutor_service.py, and tutor_lesson_utils.py"
    },
    {
        "question": "What files does tutor_service.py depend on?",
        "ground_truth": "tutor_service.py depends on neo4j_conn.py, student_db.py, diagnoser_service.py, intent_service.py, planner_service.py, and tutor_lesson_utils.py"
    },

    # --- VECTOR / IMPLEMENTATION ---
    {
        "question": "What is the purpose of StudentDB class?",
        "ground_truth": "StudentDB manages student session persistence using TinyDB, storing mastery scores, planned learning paths, progress tracking, and last JSON path for each student identified by student_id"
    },
    {
        "question": "How does app.py initialize its services?",
        "ground_truth": "app.py initializes PDFService, LLMService, Neo4jConn, GraphService, and TutorWorkflow in Streamlit session state to avoid re-initialization on every rerun"
    },
    {
        "question": "What happens when a user uploads a PDF in app.py?",
        "ground_truth": "The PDF is saved temporarily, converted to markdown using PDFService, split into chunks, analyzed for concepts using LLMService, saved as JSON, loaded into Neo4j via GraphService, and the student session is updated"
    },
    {
        "question": "What is the role of TutorWorkflow?",
        "ground_truth": "TutorWorkflow orchestrates the multi-agent learning session using LangGraph StateGraph with nodes for intent parsing, planning, teaching, diagnosis, and evaluation"
    },
    {
        "question": "How does the tutor handle an already active session vs a new session?",
        "ground_truth": "If state.values and state.next exist, the tutor resumes via Command(resume=user_input). Otherwise it starts fresh by initializing a new GraphState with all default values"
    },
    {
        "question": "What is GraphState and what fields does it contain?",
        "ground_truth": "GraphState is a TypedDict that holds student_id, messages, current_input, target_topics, known_topics, current_concept, current_question, student_answer, answer_score, diagnosis_report, planned_paths, current_path_index, current_concept_index, final_response, is_transition, and phase"
    },
    {
        "question": "What does _ensure_student_exists do in StudentDB?",
        "ground_truth": "It checks if a student record exists in TinyDB, and if not, inserts a new record with default values for mastery, planned_path, progress, last_json_path, created_at, and last_updated"
    },

    # --- HYBRID ---
    {
        "question": "How does app.py use StudentDB?",
        "ground_truth": "app.py imports StudentDB from src/database and uses it to retrieve last_json_path, update session data, and manage student state across learning sessions"
    },
    {
        "question": "How does the planner node work in TutorWorkflow?",
        "ground_truth": "The planner node retrieves student mastery from StudentDB, uses PlannerService with MSMS algorithm to find optimal learning paths, saves the path, and routes to tutor_teach node or END if no paths are found"
    },
    {
        "question": "What LLM models are used in TutorWorkflow and for what purpose?",
        "ground_truth": "TutorWorkflow uses two Groq Llama-3.3-70b instances — self.llm with max_tokens=250 for quick responses like quiz questions, and self.teach_llm with max_tokens=1000 and temperature=0.35 for longer concept explanations"
    },
    {
        "question": "How does app.py handle the case where no concepts are identified from a PDF?",
        "ground_truth": "If the concepts list is empty after processing, app.py updates the status to error state, shows an error message asking for a more detailed document, and calls st.stop()"
    },
    {
        "question": "What is the entry point of the LangGraph workflow in TutorWorkflow?",
        "ground_truth": "The entry point is the intent_parser node, set via builder.set_entry_point('intent_parser')"
    },
]

In [79]:
results = []

for sample in eval_questions:
    raw_result = engine.get_result(sample["question"], top_k=5)
    
    context = formate_documents(raw_result)
    
    response = chat_engine.generate_response(sample["question"], context)
    
    chunks = []
    if raw_result["type"] in ("graph", "hybrid"):
        chunks.append(str(raw_result["graph"]))
    if raw_result["type"] == "hybrid":
        chunks.extend(raw_result["vector"]["documents"][0])

    results.append({
        "question": sample["question"],
        "answer": response.answer,
        "contexts": chunks,
        "ground_truth": sample["ground_truth"]
    })

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kkvspkhpfydb3gfhbgqcfryf` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99672, Requested 580. Please try again in 3m37.728s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm
)

print(scores)

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]Exception raised in Job[11]: BadRequestError(Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}})
Exception raised in Job[6]: BadRequestError(Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}})
Evaluating:   1%|          | 1/90 [00:00<01:10,  1.26it/s]Exception raised in Job[8]: BadRequestError(Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}})
Exception raised in Job[4]: BadRequestError(Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}})
Exception raised in Job[13]: BadRequestError(Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}})
Evaluating:   6%|▌         | 5/90 [00:00<00:12,  6

{'faithfulness': nan, 'answer_relevancy': nan, 'answer_correctness': nan, 'context_precision': nan, 'context_recall': nan}


In [39]:
from typing import TypedDict

class QueryState(TypedDict):
    question: str
    vector_context: str
    graph_context: str
    agent: str
    answer: str
